# Latest LeiData WAV 1 dictionary-learning results

This notebook presents the most recent output produced by `run_build_dictionary_leidata_wav1.sh`. It resolves the latest UTC-stamped run directory (via the catalog, or the newest `YYYYMMDDTHHMMSSZ/` child of the result base), loads `manifest.json`, checks the corresponding catalog record, summarizes every outer round, visualizes energy reduction and coverage, displays the learned templates, and examines their matchings.

Re-run all cells after invoking the shell script (and `python -m database.migrate_flat_dict_dirs` once if older flat artifacts still sit at the result base) to refresh the presentation.

In [ ]:
%matplotlib inline

from collections import Counter, defaultdict
from pathlib import Path
import json
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
RESULT_BASE = LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_averaged"
RUN_STAMP_RE = __import__("re").compile(r"^\d{8}T\d{6}Z$")


def _resolve_latest_run_dir(base: Path, db_path: Path) -> tuple[Path, dict]:
    """Prefer newest completed catalog run under ``base``; else newest stamped child."""
    base = base.resolve()
    run = None
    if db_path.is_file():
        with sqlite3.connect(db_path) as conn:
            conn.row_factory = sqlite3.Row
            run = conn.execute(
                """
                SELECT * FROM learning_runs
                WHERE status = 'completed'
                  AND (out_dir = ? OR out_dir LIKE ?)
                ORDER BY run_id DESC
                LIMIT 1
                """,
                (str(base), str(base) + "/%"),
            ).fetchone()
            if run is not None:
                run = dict(run)
                run_dir = Path(run["out_dir"])
                if (run_dir / "manifest.json").is_file():
                    return run_dir, run

    stamped = sorted(
        (p for p in base.iterdir() if p.is_dir() and RUN_STAMP_RE.fullmatch(p.name)),
        key=lambda p: p.name,
    )
    if stamped and (stamped[-1] / "manifest.json").is_file():
        return stamped[-1], run or {}
    flat_manifest = base / "manifest.json"
    if flat_manifest.is_file():
        return base, run or {}
    raise FileNotFoundError(
        f"No stamped run dir or flat manifest under {base}. "
        "Run run_build_dictionary_leidata_wav1.sh (and migrate_flat_dict_dirs if needed)."
    )


RESULT_DIR, run = _resolve_latest_run_dir(RESULT_BASE, DB_PATH)
MANIFEST_PATH = RESULT_DIR / "manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text())
rounds = sorted(manifest["round_logs"], key=lambda row: row["round_index"])
templates = {
    item["template_id"]: np.load(RESULT_DIR / item["file"])
    for item in manifest["templates"]
}
matchings = manifest["matchings"]

if not run and DB_PATH.is_file():
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT * FROM learning_runs WHERE out_dir = ? ORDER BY run_id DESC LIMIT 1",
            (str(RESULT_DIR.resolve()),),
        ).fetchone()
        run = dict(row) if row is not None else {"run_id": "?", "status": "unknown", "finished_at": None}

print(f"Loaded run {run.get('run_id')} ({run.get('status')})")
print(f"Result directory: {RESULT_DIR}")
print(f"Finished: {run.get('finished_at')}")

## Run summary

The summary below is generated from the manifest and catalog, so it updates when the script is run again. Energy reduction is reported both absolutely and as a fraction of the initial residual energy.

In [ ]:
energy_before = float(manifest["energy_before"])
energy_after = float(manifest["energy_after"])
energy_reduction = energy_before - energy_after
energy_reduction_pct = 100 * energy_reduction / energy_before
params = manifest["params"]

summary_rows = [
    ("Run", run["run_id"]),
    ("Status", run["status"]),
    ("WAV", Path(params["wav"]).name),
    ("Windows", len(manifest["window_ids"])),
    ("Outer rounds completed", len(rounds)),
    ("Templates", len(templates)),
    ("Final matchings", len(matchings)),
    ("Initial energy", f"{energy_before:,.0f}"),
    ("Final energy", f"{energy_after:,.0f}"),
    ("Energy removed", f"{energy_reduction:,.0f} ({energy_reduction_pct:.1f}%)"),
    ("Final window coverage", f"{rounds[-1]['window_coverage_fraction']:.1%}"),
    ("Final time-bin coverage", f"{rounds[-1]['time_coverage_fraction']:.1%}"),
    ("Stop reason", manifest["stop_reason"]),
]
summary_md = "| Metric | Value |\n|---|---|\n" + "\n".join(
    f"| {label} | {value} |" for label, value in summary_rows
)
display(Markdown(summary_md))

param_md = "| Parameter | Value |\n|---|---:|\n" + "\n".join(
    f"| `{key}` | `{value}` |" for key, value in params.items()
)
display(Markdown("### Parameters\n" + param_md))

## Results by outer round

Each row compares the winning seed's matches with the matches after averaging and re-running matching pursuit. Coverage is cumulative: window coverage asks whether each input window has at least one match, while time-bin coverage measures the union of matched template intervals.

In [ ]:
round_header = (
    "| Round | Template | Width | Seed matches | Averaged matches | "
    "Energy removed | Cumulative removed | Windows | Time bins |\n"
    "|---:|---:|---:|---:|---:|---:|---:|---:|---:|\n"
)
round_lines = []
for row in rounds:
    round_lines.append(
        f"| {row['round_index']} | {row['template_id']} | {row['template_width']} | "
        f"{row['seed_n_matchings']} | {row['averaged_n_matchings']} | "
        f"{row['delta_energy']:,.0f} | {row['cumulative_delta_energy']:,.0f} | "
        f"{row['window_coverage_fraction']:.1%} | {row['time_coverage_fraction']:.1%} |"
    )
display(Markdown(round_header + "\n".join(round_lines)))

round_number = np.array([row["round_index"] + 1 for row in rounds])
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

axes[0, 0].plot(
    np.r_[0, round_number],
    np.r_[energy_before, [row["energy_after"] for row in rounds]],
    marker="o",
)
axes[0, 0].set(title="Residual energy", xlabel="Completed rounds", ylabel="Energy")
axes[0, 0].grid(alpha=0.25)

axes[0, 1].bar(round_number, [row["delta_energy"] for row in rounds])
axes[0, 1].set(title="Energy removed by each averaged template", xlabel="Round", ylabel="Δ energy")
axes[0, 1].grid(axis="y", alpha=0.25)

width = 0.36
axes[1, 0].bar(round_number - width / 2, [row["seed_n_matchings"] for row in rounds], width, label="Winning seed")
axes[1, 0].bar(round_number + width / 2, [row["averaged_n_matchings"] for row in rounds], width, label="After averaging")
axes[1, 0].set(title="Match count", xlabel="Round", ylabel="Matches")
axes[1, 0].legend()
axes[1, 0].grid(axis="y", alpha=0.25)

axes[1, 1].plot(round_number, [100 * row["window_coverage_fraction"] for row in rounds], marker="o", label="Windows")
axes[1, 1].plot(round_number, [100 * row["time_coverage_fraction"] for row in rounds], marker="o", label="Time bins")
axes[1, 1].set(title="Cumulative coverage", xlabel="Round", ylabel="Coverage (%)", ylim=(0, 105))
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.25)

plt.show()

## Learned templates

Each image is one dictionary atom. Color limits are symmetric around zero and independently scaled to the 99th percentile of absolute magnitude, making weak and strong templates visually comparable. The vertical axis is the spectrogram frequency-bin index and the horizontal axis is template time-bin offset.

In [ ]:
n_templates = len(templates)
matching_counts = Counter(item["template_id"] for item in matchings)
n_cols = min(5, max(1, n_templates))
n_rows = int(np.ceil(n_templates / n_cols))
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.0 * n_cols, 3.4 * n_rows),
    squeeze=False,
    constrained_layout=True,
)

for ax in axes.flat[n_templates:]:
    ax.axis("off")

for ax, (template_id, template) in zip(axes.flat, sorted(templates.items())):
    vmax = float(np.percentile(np.abs(template), 99))
    if vmax == 0:
        vmax = 1.0
    image = ax.imshow(template, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(
        f"T{template_id}  {template.shape[0]}×{template.shape[1]}\n"
        f"matches={matching_counts[template_id]}"
    )
    ax.set_xlabel("Time")
    ax.set_ylabel("Freq")
    fig.colorbar(image, ax=ax, orientation="horizontal", pad=0.08, fraction=0.05)

plt.show()

## Final matching distribution

These are the final matchings saved with the dictionary (unrefined if global refine aborted). Counts are split by input window; amplitude distributions retain sign because matching pursuit can use both positive and negative template coefficients.

In [ ]:
short_window = {
    window_id: f"window {index + 1}"
    for index, window_id in enumerate(manifest["window_ids"])
}
counts_by_window = defaultdict(Counter)
for item in matchings:
    counts_by_window[item["template_id"]][short_window[item["window_id"]]] += 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
window_names = list(short_window.values())
bottom = np.zeros(n_templates)
for window_name in window_names:
    values = np.array([
        counts_by_window[template_id][window_name]
        for template_id in sorted(templates)
    ])
    axes[0].bar(sorted(templates), values, bottom=bottom, label=window_name)
    bottom += values
axes[0].set(title="Matches by template and window", xlabel="Template", ylabel="Match count", xticks=sorted(templates))
axes[0].legend()
axes[0].grid(axis="y", alpha=0.25)

amplitudes = [
    [item["amplitude"] for item in matchings if item["template_id"] == template_id]
    for template_id in sorted(templates)
]
axes[1].boxplot(amplitudes, tick_labels=[str(i) for i in sorted(templates)], showfliers=True)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Matching amplitudes", xlabel="Template", ylabel="Amplitude")
axes[1].grid(axis="y", alpha=0.25)

plt.show()

## Matches on top of the original spectrograms

Each row shows an **original input spectrogram** loaded from the catalog, before any template subtraction. The grayscale layer is deliberately faint over a white background so the colored detections remain prominent.

Colored bands mark where each final template matched. Labels identify the template; solid edges indicate positive amplitudes and dashed edges indicate negative amplitudes.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    window_paths = {
        row["window_id"]: row["npy_path"]
        for row in conn.execute(
            f"SELECT window_id, npy_path FROM windows WHERE window_id IN "
            f"({','.join('?' * len(manifest['window_ids']))})",
            manifest["window_ids"],
        )
    }

original_spectrograms = {
    window_id: np.asarray(np.load(window_paths[window_id]), dtype=np.float64)
    for window_id in manifest["window_ids"]
}

print(
    "Loaded original spectrograms: "
    + ", ".join(
        f"{short_window[window_id]}={array.shape}"
        for window_id, array in original_spectrograms.items()
    )
)

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

template_ids = sorted(templates)
color_cycle = plt.cm.tab20(np.linspace(0, 1, max(20, len(template_ids))))
template_colors = {
    template_id: color_cycle[index % len(color_cycle)]
    for index, template_id in enumerate(template_ids)
}
matches_by_window = defaultdict(list)
for item in matchings:
    matches_by_window[item["window_id"]].append(item)

n_windows_plot = len(manifest["window_ids"])
fig, axes = plt.subplots(
    n_windows_plot, 1, figsize=(16, 5.5 * n_windows_plot), squeeze=False, constrained_layout=True
)

for ax, window_id in zip(axes.flat, manifest["window_ids"]):
    original = original_spectrograms[window_id]
    vmin, vmax = np.percentile(original, [5, 99.5])
    ax.set_facecolor("white")
    ax.imshow(
        original,
        aspect="auto",
        origin="lower",
        cmap="Greys",
        vmin=vmin,
        vmax=vmax,
        alpha=0.22,
    )

    n_freq = original.shape[0]
    for item in sorted(matches_by_window[window_id], key=lambda m: m["t_start"]):
        template_id = item["template_id"]
        width = templates[template_id].shape[1]
        color = template_colors[template_id]
        ax.add_patch(
            mpatches.Rectangle(
                (item["t_start"] - 0.5, -0.5),
                width,
                n_freq,
                facecolor=color,
                alpha=0.30,
                edgecolor=color,
                linewidth=1.8,
                linestyle="-" if item["amplitude"] >= 0 else "--",
            )
        )
        ax.annotate(
            str(template_id),
            (item["t_start"] + width / 2, n_freq * 0.97),
            color=color,
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
        )

    ax.set_title(
        f"{short_window[window_id]} — original spectrogram with "
        f"{len(matches_by_window[window_id])} matches ({window_id})"
    )
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Frequency bin")

legend_handles = [
    mpatches.Patch(
        facecolor=template_colors[template_id],
        alpha=0.4,
        edgecolor=template_colors[template_id],
        label=f"template {template_id} (w={templates[template_id].shape[1]}, "
        f"{matching_counts[template_id]} matches)",
    )
    for template_id in template_ids
] + [
    Line2D([], [], color="black", linestyle="-", label="positive amplitude"),
    Line2D([], [], color="black", linestyle="--", label="negative amplitude"),
]
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncols=4, framealpha=0.9)
plt.show()

In [ ]:
first_full_window_round = next(
    (row["round_index"] + 1 for row in rounds if row["window_coverage_fraction"] >= 1.0),
    None,
)
first_two_share = 100 * sum(row["delta_energy"] for row in rounds[:2]) / energy_reduction
amplitude_scale = {
    template_id: max(abs(item["amplitude"]) for item in matchings if item["template_id"] == template_id)
    for template_id in templates
}
largest_amplitude_template = max(amplitude_scale, key=amplitude_scale.get)

interpretation = f"""## Interpretation

- All **{len(manifest['window_ids'])} windows** received a match by round **{first_full_window_round}**, but only **{rounds[-1]['time_coverage_fraction']:.1%} of time bins** were covered after {len(rounds)} rounds. Window coverage therefore indicates breadth across windows, not dense temporal coverage.
- The first two templates account for **{first_two_share:.1f}%** of total energy reduction. Later rounds still added coverage (time-bin coverage ends at **{rounds[-1]['time_coverage_fraction']:.1%}**), but marginal ΔE fell steadily.
- The run learned and registered **{len(templates)} templates with {len(matchings)} final matchings**, reducing residual energy by **{energy_reduction:,.0f} ({energy_reduction_pct:.1f}%)**. Outer learning used `min_repeats={params.get('min_repeats')}` and optional `n_rounds={params.get('n_rounds')}`.
- Stop reason: `{manifest['stop_reason']}`. A `refine_aborted` result means the successfully learned, unrefined dictionary was retained and the catalog run completed normally.
- Template **{largest_amplitude_template}** has the largest absolute matching coefficient ({amplitude_scale[largest_amplitude_template]:.2f}). It is worth inspecting when comparing atom normalization, near-collinear templates, or repeated placements around the same event.
"""
display(Markdown(interpretation))